In [1]:
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler, TensorDataset
from torch.optim import Optimizer
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
import xgboost as xgb
from tqdm import tqdm

In [2]:
# Custom Dataset
class MortgageDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


# Custom Optimizer
class AdvancedSymplecticOptimizer(Optimizer):
    def __init__(self, params, lr=1e-3, beta=0.9, epsilon=1e-8):
        defaults = dict(lr=lr, beta=beta, epsilon=epsilon)
        super(AdvancedSymplecticOptimizer, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['momentum'] = torch.zeros_like(p.data)

                momentum = state['momentum']
                lr, beta, eps = group['lr'], group['beta'], group['epsilon']

                state['step'] += 1

                # Implement 4th order symplectic integrator (Forest-Ruth algorithm)
                momentum.mul_(beta).add_(grad, alpha=1 - beta)

                # Compute adaptive step size
                kinetic = 0.5 * (momentum ** 2).sum()
                potential = 0.5 * (grad ** 2).sum()
                hamiltonian = kinetic + potential
                step_size = lr / (hamiltonian.sqrt() + eps)

                p.add_(momentum, alpha=-step_size)

        return loss


class HamiltonianNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], activation='relu', dropout_rate=0.0):
        super().__init__()
        self.layers = nn.ModuleList()

        # Input layer
        self.layers.append(nn.Linear(input_dim, hidden_dims[0]))

        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            self.layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))

        # Output layer
        self.layers.append(nn.Linear(hidden_dims[-1], 2))  # Binary classification

        # Activation function
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'leaky_relu':
            self.activation = nn.LeakyReLU()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError("Unsupported activation function")

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        for layer in self.layers[:-1]:  # All layers except the last
            x = self.dropout(self.activation(layer(x)))
        return self.layers[-1](x)  # Last layer (no activation for output)


# Hamiltonian-inspired loss function
def hamiltonian_loss(outputs, labels, model, reg_coeff=0.01):
    loss_fct = nn.CrossEntropyLoss()
    base_loss = loss_fct(outputs, labels)
    # Add regularization based on Hamiltonian principles
    param_norm = sum(p.norm().item() for p in model.parameters())
    reg_term = reg_coeff * param_norm  # Use reg_coeff instead of a fixed value
    return base_loss + reg_term


# Evaluation function
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    auc = roc_auc_score(all_labels, all_preds)

    return accuracy, precision, recall, f1, auc

In [3]:
# Load the data
train_df = pd.read_csv("training.csv")
oot_test_df = pd.read_csv("OOT test.csv")
train_df = train_df.dropna()
oot_test_df = oot_test_df.dropna()

# Prepare features and target
features = ['Credit_Score', 'Mortgage_Insurance', 'Number_of_units', 'CLoan_to_value',
            'Debt_to_income', 'OLoan_to_value', 'Single_borrower']

X_train = train_df[features]
y_train = train_df['DFlag']
X_oot_test = oot_test_df[features]
y_oot_test = oot_test_df['DFlag']

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_oot_test_scaled = scaler.transform(X_oot_test)

# Apply SMOTE to balance the training dataset
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_resampled)
y_train_tensor = torch.LongTensor(y_train_resampled)
X_oot_test_tensor = torch.FloatTensor(X_oot_test_scaled)
y_oot_test_tensor = torch.LongTensor(y_oot_test.values)

# Create DataLoader for training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Create DataLoader for OOT testing
oot_test_dataset = TensorDataset(X_oot_test_tensor, y_oot_test_tensor)
oot_test_loader = DataLoader(oot_test_dataset, batch_size=32, shuffle=False)

# Print class distribution before and after SMOTE
print("Class distribution before SMOTE:", np.bincount(y_train))
print("Class distribution after SMOTE:", np.bincount(y_train_resampled))

Class distribution before SMOTE: [230460    591]
Class distribution after SMOTE: [230460 230460]


In [4]:
# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Custom LR Scheduler
class CustomLRScheduler:
    def __init__(self, optimizer, factor=0.5, patience=3, min_lr=1e-5):
        self.optimizer = optimizer
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.best_loss = float('inf')
        self.bad_epochs = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1

        if self.bad_epochs > self.patience:
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = max(param_group['lr'] * self.factor, self.min_lr)
            self.bad_epochs = 0
            print(f"Learning rate reduced to {param_group['lr']}")

In [10]:
# Ensure X_train_tensor and y_train_tensor are PyTorch tensors
if not isinstance(X_train_tensor, torch.Tensor):
    X_train_tensor = torch.tensor(X_train_tensor, dtype=torch.float32)
if not isinstance(y_train_tensor, torch.Tensor):
    y_train_tensor = torch.tensor(y_train_tensor, dtype=torch.long)

# Split the training data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor.numpy(), y_train_tensor.numpy(),
    test_size=0.2, random_state=42
)

# Convert back to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

# Create DataLoaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# Hyperparameter search space
hidden_dims_options = [[64, 32], [128, 64], [256, 128, 64]]
activations = ['relu', 'leaky_relu', 'tanh']
dropout_rates = [0.0, 0.2, 0.4]
learning_rates = [1e-4, 1e-3, 1e-2]
reg_coeffs = [0.001, 0.01, 0.1]

best_params = None
best_performance = 0

for hidden_dims, activation, dropout_rate, lr, reg_coeff in itertools.product(
    hidden_dims_options, activations, dropout_rates, learning_rates, reg_coeffs):

    model = HamiltonianNN(
        input_dim=X_train.shape[1],
        hidden_dims=hidden_dims,
        activation=activation,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = AdvancedSymplecticOptimizer(model.parameters(), lr=lr)
    scheduler = CustomLRScheduler(optimizer)
    num_epochs = 10

    # Training loop
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = hamiltonian_loss(outputs, labels, model, reg_coeff)
            loss.backward()
            optimizer.step()

    # Evaluation
    val_accuracy, _, _, val_f1, _ = evaluate_model(model, val_loader, device)

    if val_f1 > best_performance:
        best_performance = val_f1
        best_params = {
            'hidden_dims': hidden_dims,
            'activation': activation,
            'dropout_rate': dropout_rate,
            'lr': lr,
            'reg_coeff': reg_coeff
        }

print("Best hyperparameters:", best_params)
print("Best validation F1 score:", best_performance)

# Train final model with best hyperparameters
model = HamiltonianNN(
    input_dim=X_train.shape[1],
    hidden_dims=best_params['hidden_dims'],
    activation=best_params['activation'],
    dropout_rate=best_params['dropout_rate']
).to(device)


# K-fold Cross-validation
k_folds = 3
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

cv_accuracies = []
cv_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
    print(f"\nFold {fold+1}/{k_folds}")

    X_train_fold, X_val_fold = X_train_tensor[train_idx], X_train_tensor[val_idx]
    y_train_fold, y_val_fold = y_train_tensor[train_idx], y_train_tensor[val_idx]

    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)

    model = HamiltonianNN(input_dim=X_train.shape[1]).to(device)
    optimizer = AdvancedSymplecticOptimizer(model.parameters(), lr=0.001)
    scheduler = CustomLRScheduler(optimizer)

    best_val_loss = float('inf')
    best_model_state = None

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for features, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            features, labels = features.to(device), labels.to(device)

            outputs = model(features)
            loss = hamiltonian_loss(outputs, labels, model)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Average Train Loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += hamiltonian_loss(outputs, labels, model).item()

        val_loss /= len(val_loader)
        val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate_model(model, val_loader, device)
        print(f"Validation - Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

        # Update learning rate
        scheduler.step(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            print("New best model saved")

    # Load best model for final evaluation
    model.load_state_dict(best_model_state)
    val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate_model(model, val_loader, device)
    cv_accuracies.append(val_accuracy)
    cv_f1_scores.append(val_f1)
    print(f"Final Validation - Accuracy: {val_accuracy:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

# Load best model for final evaluation
model.load_state_dict(best_model_state)
print("Training completed. Best model loaded.")

val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate_model(model, val_loader, device)
cv_accuracies.append(val_accuracy)
cv_f1_scores.append(val_f1)
print(f"Final Validation - Accuracy: {val_accuracy:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

KeyboardInterrupt: 

In [ ]:
print("\nCross-validation results:")
print(f"Mean Accuracy: {np.mean(cv_accuracies):.4f} (+/- {np.std(cv_accuracies):.4f})")
print(f"Mean F1 Score: {np.mean(cv_f1_scores):.4f} (+/- {np.std(cv_f1_scores):.4f})")
print(f"Mean AUC Score: {np.mean(val_auc):.4f} (+/- {np.std(val_auc):.4f})")

In [ ]:
# Train on full training set
full_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)  # Use X_train_tensor instead of X_train_val
full_train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True)
model = HamiltonianNN(input_dim=len(features)).to(device)
optimizer = AdvancedSymplecticOptimizer(model.parameters(), lr=0.001)
scheduler = CustomLRScheduler(optimizer)

best_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for features, labels in tqdm(full_train_loader, desc=f"Full Training - Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        features, labels = features.to(device), labels.to(device)

        outputs = model(features)
        loss = hamiltonian_loss(outputs, labels, model)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(full_train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Train Loss: {avg_train_loss:.4f}")

    # Validation on OOT test set
    model.eval()
    oot_loss = 0
    oot_dataset = TensorDataset(X_oot_test_tensor, y_oot_test_tensor)  # Use OOT test data
    oot_loader = DataLoader(oot_dataset, batch_size=32)

    with torch.no_grad():
        for features, labels in oot_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            oot_loss += hamiltonian_loss(outputs, labels, model).item()

    oot_loss /= len(oot_loader)
    print(f"OOT Test Loss: {oot_loss:.4f}")

    # Update learning rate
    scheduler.step(oot_loss)

    # Save best model
    if oot_loss < best_loss:
        best_loss = oot_loss
        best_model_state = model.state_dict().copy()
        print("New best model saved")

# Load best model for final evaluation
model.load_state_dict(best_model_state)
print("Training completed. Best model loaded.")

In [ ]:
oot_test_dataset = TensorDataset(X_oot_test_tensor, y_oot_test_tensor)
oot_test_loader = DataLoader(oot_test_dataset, batch_size=32, shuffle=False)

oot_accuracy, oot_precision, oot_recall, oot_f1, oot_auc = evaluate_model(model, oot_test_loader, device)

print("\nFinal OOT Test Results:")
print(f"Accuracy: {oot_accuracy:.4f}, Precision: {oot_precision:.4f}, Recall: {oot_recall:.4f}")
print(f"F1: {oot_f1:.4f}, AUC: {oot_auc:.4f}")

## Using XGBoost

In [ ]:
"""# Split the resampled training data into training and validation sets
X_train_final, X_val, y_train_final, y_val = train_test_split(X_train_resampled, y_train_resampled, test_size=0.2, random_state=42)

# Define XGBoost model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Define parameter grid for GridSearchCV
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5]
}

# Perform GridSearchCV
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid,
                           cv=3, scoring='roc_auc', verbose=2, n_jobs=-1)
grid_search.fit(X_train_final, y_train_final)

# Get the best model
best_xgb_model = grid_search.best_estimator_

# Train the best model on the entire training set
best_xgb_model.fit(X_train_resampled, y_train_resampled)

# Make predictions on the OOT test set
y_oot_pred = best_xgb_model.predict(X_oot_test_scaled)
y_oot_pred_proba = best_xgb_model.predict_proba(X_oot_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_oot_test, y_oot_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_oot_test, y_oot_pred, average='weighted', zero_division=0)
auc = roc_auc_score(y_oot_test, y_oot_pred_proba)

# Print results
print("\nXGBoost Final OOT Test Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

# Print best parameters
print("\nBest XGBoost Parameters:")
print(grid_search.best_params_)

# Feature importance
feature_importance = best_xgb_model.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + .5

print("\nFeature Importances:")
for idx in sorted_idx:
    print(f"{features[idx]}: {feature_importance[idx]:.4f}")"""